In [1]:
pip install langchain-openai langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Fo

In [2]:
from langchain_openai import OpenAIEmbeddings

# 임베딩 모델 초기화
embeddings = OpenAIEmbeddings()  # 기본: text-embedding-3-small

# 텍스트를 벡터로 변환
vector = embeddings.embed_query("LangGraph란 무엇인가요?")
print(f"벡터 차원: {len(vector)}")  # 1536
print(f"벡터 샘플: {vector[:5]}")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 문서 로드 및 청킹
loader = PyPDFLoader("./data/manual.pdf")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = splitter.split_documents(documents)

# 2. 벡터스토어 생성 (임베딩 + 저장을 한 번에)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(),
    collection_name="manual_docs",
    persist_directory="./chroma_db",    # 디스크에 영속 저장
)

print(f"저장된 문서 수: {vectorstore._collection.count()}")


In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 기존 벡터스토어 로드 — embedding_function 필수!
vectorstore = Chroma(
    collection_name="manual_docs",
    embedding_function=OpenAIEmbeddings(),
    persist_directory="./chroma_db",
)

print(f"로드된 문서 수: {vectorstore._collection.count()}")


In [ ]:
# 유사도 점수 포함 검색
results_with_scores = vectorstore.similarity_search_with_score(
    "LangGraph 상태 관리", k=3
)

for doc, score in results_with_scores:
    print(f"점수: {score:.4f} | {doc.page_content[:80]}...")

In [ ]:
# 기본 Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Retriever로 검색
docs = retriever.invoke("LangGraph 노드 구현 방법")
print(f"검색된 문서 수: {len(docs)}")

# MMR (Maximal Marginal Relevance) — 다양성 확보
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 10},
)

docs = mmr_retriever.invoke("LangGraph 노드 구현 방법")